# BFC Competitor Check 2026
## Notebook 03: Data Cleaning

### Purpose

Validate and prepare the collected datasets for analysis while
preserving the original evidence and collection limitations.

### Scope

Check table structure, identifiers, missing values, duplicates,
data types and relationships between datasets.

Collection remains partial across all ten research questions.
Cleaning does not resolve missing evidence or establish comparability.

### Rules

1. Preserve raw files.
2. Keep unsupported values missing.
3. Retain source references and evidence qualifications.
4. Inspect possible duplicates before removing records.
5. Keep promotional prices separate from other prices.
6. Keep provisional branch identities unresolved until verified.
7. Save prepared datasets separately after validation.

### Study baseline

October 23, 2026.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# 1. Locate the project.

project_root = Path.cwd().resolve()

while (
    not (project_root / "references" / "source_register.csv").is_file()
    and project_root != project_root.parent
):
    project_root = project_root.parent

if not (project_root / "references" / "source_register.csv").is_file():
    raise FileNotFoundError("Run this notebook from inside the project.")

raw_data_path = project_root / "data" / "raw"
processed_data_path = project_root / "data" / "processed"
references_path = project_root / "references"


# 2. Define each dataset and its unique identifier.

dataset_ids = {
    "competitor_master": "brand_id",
    "branch_directory": "branch_id",
    "menu_pricing": "observation_id",
    "promotions": "promotion_id",
    "loyalty_programs": "program_version_id",
    "store_assessments": "assessment_id",
    "customer_reviews": "review_id",
    "ordering_channels": "channel_observation_id",
    "franchise_offers": "package_version_id",
    "commercial_developments": "development_id",
    "event_packages": "event_package_id",
    "source_register": "source_id",
}


# 3. Load datasets as strings to preserve recorded values.

datasets = {}
inventory_rows = []

for dataset_name, id_column in dataset_ids.items():
    file_path = raw_data_path / f"{dataset_name}.csv"

    if dataset_name == "source_register":
        file_path = references_path / "source_register.csv"

    if not file_path.is_file():
        raise FileNotFoundError(file_path)

    df = pd.read_csv(file_path, dtype="string")
    datasets[dataset_name] = df

    inventory_rows.append({
        "dataset": dataset_name,
        "records": len(df),
        "columns": len(df.columns),
        "id_column": id_column,
        "id_column_present": id_column in df.columns,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    })

inventory = pd.DataFrame(inventory_rows)

display(inventory)


# 4. Display column names before applying cleaning rules.

for dataset_name, df in datasets.items():
    print(f"\n{dataset_name}")
    print(df.columns.tolist())

print("\nDatasets loaded:", len(datasets))
print("No files created or changed.")

,dataset,records,columns,id_column,id_column_present,exact_duplicate_rows
0,competitor_master,23,19,brand_id,True,0
1,branch_directory,24,15,branch_id,True,0
2,menu_pricing,246,21,observation_id,True,0
3,promotions,3,21,promotion_id,True,0
4,loyalty_programs,4,16,program_version_id,True,0
5,store_assessments,3,21,assessment_id,True,0
6,customer_reviews,36,16,review_id,True,0
7,ordering_channels,6,20,channel_observation_id,True,0
8,franchise_offers,15,27,package_version_id,True,0
9,commercial_developments,7,15,development_id,True,0



competitor_master
['brand_id', 'brand_name', 'operator_name', 'comparison_role', 'primary_category', 'customer_competitor', 'franchise_investment_competitor', 'official_website', 'global_founding_year', 'ph_start_year', 'price_positioning', 'ph_franchise_status', 'geographic_coverage', 'source_ids', 'verification_status', 'inclusion_status', 'inclusion_reason', 'last_verified_date', 'notes']

branch_directory
['branch_id', 'brand_id', 'branch_name', 'address', 'city', 'province', 'region', 'island_group', 'store_format', 'operating_status', 'status_date', 'observed_date', 'source_ids', 'verification_status', 'notes']

menu_pricing
['observation_id', 'brand_id', 'branch_id', 'source_id', 'observed_date', 'channel', 'location_scope', 'product_name', 'product_category', 'comparison_group', 'temperature', 'size_label', 'volume_ml', 'regular_price_php', 'promo_price_php', 'availability_status', 'notes', 'displayed_price_php', 'price_basis', 'published_menu_section', 'product_description']


## 2. Identifier and Missing Value Checks

### Purpose

Identify missing or repeated identifiers and measure missing values
before changing any data.

### Method

Use temporary copies to treat empty or whitespace only cells as missing.
Preserve the loaded datasets and original files.

### Interpretation

Missing identifiers require investigation.

Missing evidence fields remain unknown. They will not automatically
be filled with zero, averages or assumed values.

In [2]:
# 1. Check identifiers and whitespace using temporary copies.

identifier_rows = []
identifier_issues = []
missing_value_tables = []
whitespace_rows = []

for dataset_name, df in datasets.items():
    id_column = dataset_ids[dataset_name]

    # Temporary copy for checking only.
    check_df = df.copy()

    whitespace_cells = 0

    for column in check_df.columns:
        original_values = check_df[column]
        stripped_values = original_values.str.strip()

        whitespace_cells += int(
            original_values.ne(stripped_values).fillna(False).sum()
        )

        check_df[column] = stripped_values.replace("", pd.NA)

    whitespace_rows.append({
        "dataset": dataset_name,
        "cells_with_outer_whitespace": whitespace_cells,
    })

    # Missing IDs and every occurrence of a repeated nonmissing ID.
    missing_id = check_df[id_column].isna()

    repeated_id = (
        check_df[id_column].notna()
        & check_df[id_column].duplicated(keep=False)
    )

    identifier_rows.append({
        "dataset": dataset_name,
        "id_column": id_column,
        "missing_ids": int(missing_id.sum()),
        "rows_with_repeated_ids": int(repeated_id.sum()),
        "distinct_repeated_ids": (
            check_df.loc[repeated_id, id_column].nunique()
        ),
    })

    issue_mask = missing_id | repeated_id

    if issue_mask.any():
        issue_rows = df.loc[issue_mask].copy()
        issue_rows.insert(0, "dataset", dataset_name)
        identifier_issues.append(issue_rows)

    # Calculate missingness after recognising blank text.
    missing_counts = check_df.isna().sum()

    missing_table = missing_counts.rename(
        "missing_count"
    ).reset_index()

    missing_table.columns = ["field", "missing_count"]
    missing_table.insert(0, "dataset", dataset_name)
    missing_table["total_records"] = len(check_df)

    if len(check_df) > 0:
        missing_table["missing_percent"] = (
            missing_table["missing_count"] / len(check_df) * 100
        ).round(1)
    else:
        missing_table["missing_percent"] = pd.NA

    missing_value_tables.append(missing_table)


# 2. Display the identifier results.

identifier_summary = pd.DataFrame(identifier_rows)

print("Identifier checks:")
display(identifier_summary)

if identifier_issues:
    print("\nRecords requiring identifier review:")

    for issue_table in identifier_issues:
        display(issue_table)
else:
    print("No missing or repeated identifiers found.")


# 3. Display whitespace and missing value results.

whitespace_summary = pd.DataFrame(whitespace_rows)

print("\nOuter whitespace checks:")
display(whitespace_summary)

missing_value_summary = pd.concat(
    missing_value_tables,
    ignore_index=True,
)

print("\nFields containing missing values:")
display(
    missing_value_summary.loc[
        missing_value_summary["missing_count"].gt(0)
    ].sort_values(
        ["dataset", "missing_percent", "field"],
        ascending=[True, False, True],
    )
)

print("\nChecks completed using temporary copies.")
print("Loaded datasets and saved files were not changed.")

Identifier checks:


,dataset,id_column,missing_ids,rows_with_repeated_ids,distinct_repeated_ids
0,competitor_master,brand_id,0,0,0
1,branch_directory,branch_id,0,0,0
2,menu_pricing,observation_id,0,0,0
3,promotions,promotion_id,0,0,0
4,loyalty_programs,program_version_id,0,0,0
5,store_assessments,assessment_id,0,0,0
6,customer_reviews,review_id,0,0,0
7,ordering_channels,channel_observation_id,0,0,0
8,franchise_offers,package_version_id,0,0,0
9,commercial_developments,development_id,0,0,0


No missing or repeated identifiers found.

Outer whitespace checks:


,dataset,cells_with_outer_whitespace
0,competitor_master,0
1,branch_directory,0
2,menu_pricing,0
3,promotions,0
4,loyalty_programs,0
5,store_assessments,0
6,customer_reviews,0
7,ordering_channels,0
8,franchise_offers,0
9,commercial_developments,0



Fields containing missing values:


,dataset,field,missing_count,total_records,missing_percent
29,branch_directory,status_date,24,24,100.0
27,branch_directory,store_format,24,24,100.0
24,branch_directory,province,23,24,95.8
25,branch_directory,region,23,24,95.8
23,branch_directory,city,18,24,75.0
...,...,...,...,...,...
108,store_assessments,service_start_event,3,3,100.0
110,store_assessments,service_time_minutes,3,3,100.0
98,store_assessments,visit_time,3,3,100.0
103,store_assessments,wifi_status,3,3,100.0



Checks completed using temporary copies.
Loaded datasets and saved files were not changed.


## 3. Reference Integrity Checks

### Purpose

Check that populated brand, branch and source references point
to records in their corresponding datasets.

### Rules

Missing references are reported separately from unknown references.

A valid reference establishes that the linked record exists.
It does not confirm branch identity or factual accuracy.

No missing branch IDs will be assigned automatically.

In [3]:
# 1. Prepare valid reference values.

valid_brand_ids = datasets["competitor_master"]["brand_id"]
valid_branch_ids = datasets["branch_directory"]["branch_id"]
valid_source_ids = datasets["source_register"]["source_id"]

reference_rules = [
    ["brand_id", valid_brand_ids],
    ["branch_id", valid_branch_ids],
    ["source_id", valid_source_ids],
]

reference_summary_rows = []
reference_issue_tables = []


# 2. Check fields containing a single reference.

for dataset_name, df in datasets.items():
    record_id = dataset_ids[dataset_name]

    for reference_column, valid_values in reference_rules:
        if reference_column not in df.columns:
            continue

        # Skip each table's own primary identifier.
        if reference_column == record_id:
            continue

        values = df[reference_column].str.strip().replace("", pd.NA)

        missing_reference = values.isna()
        unknown_reference = (
            values.notna()
            & ~values.isin(valid_values)
        )

        reference_summary_rows.append({
            "dataset": dataset_name,
            "reference_field": reference_column,
            "missing_references": int(missing_reference.sum()),
            "unknown_references": int(unknown_reference.sum()),
        })

        if unknown_reference.any():
            issues = df.loc[
                unknown_reference,
                [record_id, reference_column],
            ].copy()

            issues.insert(0, "dataset", dataset_name)
            reference_issue_tables.append(issues)


# 3. Check fields containing several source IDs.

for dataset_name, df in datasets.items():
    if "source_ids" not in df.columns:
        continue

    record_id = dataset_ids[dataset_name]

    source_links = df[
        [record_id, "source_ids"]
    ].copy()

    source_links["source_id"] = source_links["source_ids"].str.split(";")

    source_links = source_links.explode(
        "source_id",
        ignore_index=True,
    )

    source_links["source_id"] = (
        source_links["source_id"]
        .str.strip()
        .replace("", pd.NA)
    )

    unknown_source = (
        source_links["source_id"].notna()
        & ~source_links["source_id"].isin(valid_source_ids)
    )

    missing_lists = (
        df["source_ids"]
        .str.strip()
        .replace("", pd.NA)
        .isna()
    )

    reference_summary_rows.append({
        "dataset": dataset_name,
        "reference_field": "source_ids",
        "missing_references": int(missing_lists.sum()),
        "unknown_references": int(unknown_source.sum()),
    })

    if unknown_source.any():
        issues = source_links.loc[
            unknown_source,
            [record_id, "source_id"],
        ].copy()

        issues.insert(0, "dataset", dataset_name)
        reference_issue_tables.append(issues)


# 4. Display findings.

reference_summary = pd.DataFrame(reference_summary_rows)

print("Reference integrity summary:")
display(reference_summary)

if reference_issue_tables:
    print("\nUnknown references requiring investigation:")

    for issue_table in reference_issue_tables:
        display(issue_table)
else:
    print("\nNo unknown brand, branch or source references found.")

print(
    "\nMissing references remain unresolved. "
    "They are not automatically classified as errors."
)
print("Loaded datasets and saved files were not changed.")

Reference integrity summary:


,dataset,reference_field,missing_references,unknown_references
0,branch_directory,brand_id,0,0
1,menu_pricing,brand_id,0,0
2,menu_pricing,branch_id,246,0
3,menu_pricing,source_id,0,0
4,promotions,brand_id,0,0
5,promotions,branch_id,3,0
6,promotions,source_id,0,0
7,loyalty_programs,brand_id,0,0
8,loyalty_programs,source_id,0,0
9,store_assessments,brand_id,0,0



No unknown brand, branch or source references found.

Missing references remain unresolved. They are not automatically classified as errors.
Loaded datasets and saved files were not changed.


In [4]:
# Check relationships using only loaded datasets and saved sample records.

review_sampling = pd.read_csv(
    references_path / "review_sampling.csv",
    dtype="string",
)

parent_tables = {
    "branch_directory": datasets["branch_directory"],
    "source_register": datasets["source_register"],
    "review_sampling": review_sampling,
    "commercial_developments": datasets["commercial_developments"],
}

# Define links present in each dataset.

relationship_rules = []

for dataset_name, df in datasets.items():
    if "branch_id" in df.columns and dataset_name != "branch_directory":
        relationship_rules.append(
            [dataset_name, "branch_id", "branch_directory", "branch_id"]
        )

    if "source_id" in df.columns and dataset_name != "source_register":
        relationship_rules.append(
            [dataset_name, "source_id", "source_register", "source_id"]
        )

    if "source_ids" in df.columns:
        relationship_rules.append(
            [dataset_name, "source_ids", "source_register", "source_id"]
        )

relationship_rules.extend(
    [
        ["customer_reviews", "sample_id", "review_sampling", "sample_id"],
        [
            "event_packages",
            "development_id",
            "commercial_developments",
            "development_id",
        ],
    ]
)

relationship_summary_rows = []
relationship_issue_rows = []

for dataset_name, link_field, parent_name, parent_key in relationship_rules:
    child = datasets[dataset_name]
    parent = parent_tables[parent_name]
    record_id = dataset_ids[dataset_name]

    if parent[parent_key].isna().any():
        raise ValueError(f"Missing parent identifiers: {parent_name}")

    if parent[parent_key].duplicated().any():
        raise ValueError(f"Repeated parent identifiers: {parent_name}")

    # Compare identity fields shared by the linked tables.
    comparison_fields = []

    for field in ["brand_id", "branch_id", "platform", "source_id"]:
        if (
            field in child.columns
            and field in parent.columns
            and field != link_field
            and field != parent_key
        ):
            comparison_fields.append(field)

    # Select each column only once.
    selected_columns = [record_id]

    for field in [link_field] + comparison_fields:
        if field not in selected_columns:
            selected_columns.append(field)

    links = child[selected_columns].copy()

    if link_field == "source_ids":
        links["linked_id"] = links[link_field].str.split(";")
        links = links.explode("linked_id", ignore_index=True)
    else:
        links["linked_id"] = links[link_field]

    links["linked_id"] = (
        links["linked_id"].str.strip().replace("", pd.NA)
    )

    populated = links["linked_id"].notna()
    known = populated & links["linked_id"].isin(parent[parent_key])
    unknown = populated & ~known

    record_mismatch = pd.Series(False, index=links.index)
    missing_comparison_values = 0

    for index in links.index[unknown]:
        relationship_issue_rows.append(
            {
                "dataset": dataset_name,
                "record_id": links.loc[index, record_id],
                "link_field": link_field,
                "linked_id": links.loc[index, "linked_id"],
                "issue": "Unknown parent reference",
                "comparison_field": pd.NA,
                "recorded_value": pd.NA,
                "parent_value": pd.NA,
            }
        )

    for field in comparison_fields:
        parent_values = links["linked_id"].map(
            parent.set_index(parent_key)[field]
        )

        missing_values = known & (
            links[field].isna() | parent_values.isna()
        )

        missing_comparison_values += int(missing_values.sum())

        mismatch = (
            known
            & links[field].notna()
            & parent_values.notna()
            & links[field].ne(parent_values).fillna(False)
        )

        record_mismatch = record_mismatch | mismatch

        for index in links.index[mismatch]:
            relationship_issue_rows.append(
                {
                    "dataset": dataset_name,
                    "record_id": links.loc[index, record_id],
                    "link_field": link_field,
                    "linked_id": links.loc[index, "linked_id"],
                    "issue": "Identity mismatch",
                    "comparison_field": field,
                    "recorded_value": links.loc[index, field],
                    "parent_value": parent_values.loc[index],
                }
            )

    relationship_summary_rows.append(
        {
            "dataset": dataset_name,
            "link_field": link_field,
            "populated_links": int(populated.sum()),
            "unknown_links": int(unknown.sum()),
            "records_with_identity_mismatch": links.loc[
                record_mismatch, record_id
            ].nunique(),
            "field_checks_missing_a_value": missing_comparison_values,
        }
    )

relationship_summary = pd.DataFrame(relationship_summary_rows)
relationship_issues = pd.DataFrame(relationship_issue_rows)

display(relationship_summary)

if relationship_issues.empty:
    print("No unknown links or identity mismatches found.")
else:
    print("Relationship issues requiring review:")
    display(relationship_issues)

print("Missing identity values do not establish agreement.")
print("No datasets or saved files changed.")

,dataset,link_field,populated_links,unknown_links,records_with_identity_mismatch,field_checks_missing_a_value
0,competitor_master,source_ids,49,0,0,0
1,branch_directory,source_ids,26,0,0,26
2,menu_pricing,branch_id,0,0,0,0
3,menu_pricing,source_id,246,0,0,246
4,promotions,branch_id,0,0,0,0
5,promotions,source_id,3,0,0,3
6,loyalty_programs,source_id,4,0,0,0
7,store_assessments,branch_id,0,0,0,0
8,store_assessments,source_id,3,0,0,3
9,customer_reviews,branch_id,36,0,0,0


No unknown links or identity mismatches found.
Missing identity values do not establish agreement.
No datasets or saved files changed.


In [5]:
# 1. Define fields that should contain numbers.

numeric_fields = {
    "competitor_master": [
        "global_founding_year",
        "ph_start_year",
    ],
    "menu_pricing": [
        "volume_ml",
        "regular_price_php",
        "promo_price_php",
        "displayed_price_php",
    ],
    "promotions": [
        "minimum_spend_php",
        "discount_value",
        "maximum_discount_php",
        "redemption_limit",
    ],
    "store_assessments": [
        "service_time_minutes",
    ],
    "customer_reviews": [
        "rating",
        "rating_scale_max",
    ],
    "ordering_channels": [
        "minimum_order_php",
        "delivery_fee_php",
        "quoted_wait_minutes",
    ],
    "franchise_offers": [
        "total_investment_min",
        "total_investment_max",
        "franchise_fee",
        "advertised_package_start_php",
    ],
    "event_packages": [
        "pax",
        "included_cups",
        "service_hours",
        "advertised_price_php",
        "source_page",
    ],
}

whole_number_fields = [
    "global_founding_year",
    "ph_start_year",
    "redemption_limit",
    "pax",
    "included_cups",
    "source_page",
]

positive_fields = [
    "volume_ml",
    "rating_scale_max",
    "pax",
    "included_cups",
    "source_page",
]

numeric_summary_rows = []
numeric_issue_tables = []
numeric_checks = {}


# 2. Parse temporary copies and identify invalid values.

for dataset_name, fields in numeric_fields.items():
    df = datasets[dataset_name]
    record_id = dataset_ids[dataset_name]

    numeric_df = pd.DataFrame(index=df.index)

    for field in fields:
        values = df[field].str.strip().replace("", pd.NA)
        numbers = pd.to_numeric(values, errors="coerce")

        numeric_df[field] = numbers

        invalid_text = values.notna() & numbers.isna()
        nonfinite = numbers.isin([float("inf"), -float("inf")])
        negative = numbers.lt(0).fillna(False)

        invalid_whole_number = pd.Series(False, index=df.index)
        invalid_zero = pd.Series(False, index=df.index)

        if field in whole_number_fields:
            finite_numbers = numbers.mask(nonfinite)
            invalid_whole_number = (
                finite_numbers.notna()
                & finite_numbers.mod(1).ne(0)
            ).fillna(False)

        if field in positive_fields:
            invalid_zero = numbers.eq(0).fillna(False)

        numeric_summary_rows.append({
            "dataset": dataset_name,
            "field": field,
            "missing_values": int(values.isna().sum()),
            "invalid_text": int(invalid_text.sum()),
            "nonfinite_values": int(nonfinite.sum()),
            "negative_values": int(negative.sum()),
            "invalid_whole_numbers": int(invalid_whole_number.sum()),
            "invalid_zero_values": int(invalid_zero.sum()),
        })

        issue_mask = (
            invalid_text
            | nonfinite
            | negative
            | invalid_whole_number
            | invalid_zero
        )

        if issue_mask.any():
            issues = df.loc[issue_mask, [record_id, field]].copy()
            issues.columns = ["record_id", "recorded_value"]
            issues.insert(0, "field", field)
            issues.insert(0, "dataset", dataset_name)
            numeric_issue_tables.append(issues)

    numeric_checks[dataset_name] = numeric_df


# 3. Check relationships between numeric fields.

range_issue_rows = []

range_rules = [
    [
        "menu_pricing",
        "promo_price_php",
        "regular_price_php",
        "Promotional price exceeds regular price",
    ],
    [
        "franchise_offers",
        "total_investment_min",
        "total_investment_max",
        "Minimum investment exceeds maximum investment",
    ],
    [
        "customer_reviews",
        "rating",
        "rating_scale_max",
        "Rating exceeds recorded scale maximum",
    ],
]

for dataset_name, lower_field, upper_field, issue in range_rules:
    numeric_df = numeric_checks[dataset_name]
    df = datasets[dataset_name]
    record_id = dataset_ids[dataset_name]

    both_present = (
        numeric_df[lower_field].notna()
        & numeric_df[upper_field].notna()
    )

    inconsistent = (
        both_present
        & numeric_df[lower_field].gt(numeric_df[upper_field])
    ).fillna(False)

    for row_index in df.index[inconsistent]:
        range_issue_rows.append({
            "dataset": dataset_name,
            "record_id": df.loc[row_index, record_id],
            "issue": issue,
            "first_value": df.loc[row_index, lower_field],
            "second_value": df.loc[row_index, upper_field],
        })


# 4. Display findings.

numeric_summary = pd.DataFrame(numeric_summary_rows)

print("Numeric validation summary:")
display(numeric_summary)

if numeric_issue_tables:
    print("\nNumeric values requiring inspection:")
    display(pd.concat(numeric_issue_tables, ignore_index=True))
else:
    print("\nNo numeric parsing or basic value issues found.")

if range_issue_rows:
    print("\nInconsistent numeric relationships:")
    display(pd.DataFrame(range_issue_rows))
else:
    print("No inconsistent numeric relationships found where both values exist.")

print("\nMissing values remain missing.")
print("Zero fees and zero prices were not automatically rejected.")
print("Large values have not been classified as errors or removed.")
print("Loaded datasets and saved files were not changed.")

Numeric validation summary:


,dataset,field,missing_values,invalid_text,nonfinite_values,negative_values,invalid_whole_numbers,invalid_zero_values
0,competitor_master,global_founding_year,18,0,0,0,0,0
1,competitor_master,ph_start_year,0,0,0,0,0,0
2,menu_pricing,volume_ml,246,0,0,0,0,0
3,menu_pricing,regular_price_php,246,0,0,0,0,0
4,menu_pricing,promo_price_php,232,0,0,0,0,0
5,menu_pricing,displayed_price_php,3,0,0,0,0,0
6,promotions,minimum_spend_php,2,0,0,0,0,0
7,promotions,discount_value,3,0,0,0,0,0
8,promotions,maximum_discount_php,3,0,0,0,0,0
9,promotions,redemption_limit,3,0,0,0,0,0



No numeric parsing or basic value issues found.
No inconsistent numeric relationships found where both values exist.

Missing values remain missing.
Zero fees and zero prices were not automatically rejected.
Large values have not been classified as errors or removed.
Loaded datasets and saved files were not changed.


In [6]:
# 1. Define fields expected to contain complete calendar dates.
# Year fields, visit times and relative review labels are excluded.

date_fields = {
    "competitor_master": ["last_verified_date"],
    "branch_directory": ["status_date", "observed_date"],
    "menu_pricing": ["observed_date"],
    "promotions": ["observed_date", "start_date", "end_date"],
    "loyalty_programs": ["observed_date", "effective_date"],
    "store_assessments": ["observed_date"],
    "customer_reviews": ["review_date", "collected_date"],
    "ordering_channels": ["observed_date"],
    "franchise_offers": ["observed_date", "effective_date"],
    "commercial_developments": [
        "observed_date",
        "published_date",
        "effective_date",
    ],
    "event_packages": ["observed_date", "effective_date"],
    "source_register": [
        "published_date",
        "effective_date",
        "collected_date",
    ],
}

baseline_date = pd.Timestamp("2026-10-23")

collection_date_fields = [
    "observed_date",
    "collected_date",
    "last_verified_date",
]

date_checks = {}
date_summary_rows = []
date_issue_tables = []


# 2. Check format and calendar validity using temporary copies.

for dataset_name, fields in date_fields.items():
    df = datasets[dataset_name]
    record_id = dataset_ids[dataset_name]

    parsed_dates = pd.DataFrame(index=df.index)

    for field in fields:
        values = df[field].str.strip().replace("", pd.NA)

        correct_format = values.str.fullmatch(
            r"\d{4}-\d{2}-\d{2}"
        ).fillna(False)

        dates = pd.to_datetime(
            values.where(correct_format),
            format="%Y-%m-%d",
            errors="coerce",
        )

        parsed_dates[field] = dates

        invalid_date = values.notna() & dates.isna()

        after_baseline = pd.Series(False, index=df.index)

        if field in collection_date_fields:
            after_baseline = dates.gt(baseline_date).fillna(False)

        date_summary_rows.append({
            "dataset": dataset_name,
            "field": field,
            "missing_dates": int(values.isna().sum()),
            "invalid_dates": int(invalid_date.sum()),
            "collection_dates_after_baseline": int(after_baseline.sum()),
        })

        issue_mask = invalid_date | after_baseline

        if issue_mask.any():
            issues = df.loc[issue_mask, [record_id, field]].copy()
            issues.columns = ["record_id", "recorded_value"]
            issues.insert(0, "field", field)
            issues.insert(0, "dataset", dataset_name)

            issues["invalid_date"] = invalid_date.loc[issue_mask].to_numpy()
            issues["after_baseline"] = after_baseline.loc[
                issue_mask
            ].to_numpy()

            date_issue_tables.append(issues)

    date_checks[dataset_name] = parsed_dates


# 3. Check clearly inconsistent date sequences.

chronology_rules = [
    ["promotions", "start_date", "end_date"],
    ["customer_reviews", "review_date", "collected_date"],
    ["source_register", "published_date", "collected_date"],
    ["commercial_developments", "published_date", "observed_date"],
]

chronology_issues = []

for dataset_name, earlier_field, later_field in chronology_rules:
    dates = date_checks[dataset_name]
    df = datasets[dataset_name]
    record_id = dataset_ids[dataset_name]

    both_present = (
        dates[earlier_field].notna()
        & dates[later_field].notna()
    )

    inconsistent = (
        both_present
        & dates[earlier_field].gt(dates[later_field])
    )

    for row_index in df.index[inconsistent]:
        chronology_issues.append({
            "dataset": dataset_name,
            "record_id": df.loc[row_index, record_id],
            "earlier_field": earlier_field,
            "earlier_value": df.loc[row_index, earlier_field],
            "later_field": later_field,
            "later_value": df.loc[row_index, later_field],
        })


# 4. Display the findings.

date_summary = pd.DataFrame(date_summary_rows)

print("Date validation summary:")
display(date_summary)

if date_issue_tables:
    print("\nDates requiring inspection:")
    display(pd.concat(date_issue_tables, ignore_index=True))
else:
    print("\nNo invalid dates or collection dates after baseline found.")

if chronology_issues:
    print("\nDate sequences requiring inspection:")
    display(pd.DataFrame(chronology_issues))
else:
    print("No inconsistent date sequences found where both dates exist.")

print("\nStudy baseline:", baseline_date.date())
print("Dates after baseline are review flags, not automatic errors.")
print("Future effective dates can describe announced offers.")
print("Relative review dates were not converted into exact dates.")
print("Loaded datasets and saved files were not changed.")

Date validation summary:


,dataset,field,missing_dates,invalid_dates,collection_dates_after_baseline
0,competitor_master,last_verified_date,3,0,0
1,branch_directory,status_date,24,0,0
2,branch_directory,observed_date,0,0,0
3,menu_pricing,observed_date,0,0,115
4,promotions,observed_date,0,0,2
5,promotions,start_date,1,0,0
6,promotions,end_date,1,0,0
7,loyalty_programs,observed_date,0,0,1
8,loyalty_programs,effective_date,4,0,0
9,store_assessments,observed_date,0,0,0



Dates requiring inspection:


,dataset,field,record_id,recorded_value,invalid_date,after_baseline
0,menu_pricing,observed_date,MNU0130,2026-10-25,False,True
1,menu_pricing,observed_date,MNU0131,2026-10-25,False,True
2,menu_pricing,observed_date,MNU0132,2026-10-25,False,True
3,menu_pricing,observed_date,MNU0133,2026-10-25,False,True
4,menu_pricing,observed_date,MNU0134,2026-10-25,False,True
...,...,...,...,...,...,...
132,source_register,collected_date,SRC0135,2026-10-25,False,True
133,source_register,collected_date,SRC0137,2026-10-25,False,True
134,source_register,collected_date,SRC0138,2026-10-25,False,True
135,source_register,collected_date,SRC0139,2026-10-25,False,True


No inconsistent date sequences found where both dates exist.

Study baseline: 2026-10-23
Dates after baseline are review flags, not automatic errors.
Future effective dates can describe announced offers.
Relative review dates were not converted into exact dates.
Loaded datasets and saved files were not changed.


In [7]:
# Inspect records with missing collection or verification dates.

sources = datasets["source_register"]
brands = datasets["competitor_master"]

missing_collection_date = (
    sources["collected_date"]
    .str.strip()
    .replace("", pd.NA)
    .isna()
)

missing_verification_date = (
    brands["last_verified_date"]
    .str.strip()
    .replace("", pd.NA)
    .isna()
)

print("Source requiring collection date review:")
with pd.option_context("display.max_colwidth", None):
    display(
        sources.loc[
            missing_collection_date,
            [
                "source_id",
                "brand_id",
                "source_title",
                "source_url",
                "collected_date",
                "access_status",
                "verification_status",
                "evidence_file",
                "notes",
            ],
        ]
    )

print("\nBrands without a verification date:")
with pd.option_context("display.max_colwidth", None):
    display(
        brands.loc[
            missing_verification_date,
            [
                "brand_id",
                "brand_name",
                "verification_status",
                "source_ids",
                "last_verified_date",
                "notes",
            ],
        ]
    )

print("\nNo dates inferred. No files changed.")

Source requiring collection date review:


,source_id,brand_id,source_title,source_url,collected_date,access_status,verification_status,evidence_file,notes
34,SRC0035,BFC021,ZUS Coffee Philippine menu,https://zuscoffee.ph/menu/,<NA>,<NA>,Pending,<NA>,"Requested menu URL redirected to https://zuscoffee.ph/rewards/. Captured content concerns rewards, not a menu. Not accepted as menu or pricing evidence."



Brands without a verification date:


,brand_id,brand_name,verification_status,source_ids,last_verified_date,notes
3,BFC004,TOMO Coffee,Pending,SRC0105;SRC0115,<NA>,"PH history review: The official About page search result states Established 2021. Direct page retrieval returned a 502 error during this check. A Bilyonaryo founder interview report states founded in 2022. Neither date is assigned as the Philippine operating start year pending clarification. Supporting URL: https://bnc.bilyonaryo.com/tomo-coffee-shifts-to-territorial-franchising-to-fuel-150-store-expansion/business/ | Invstt company profile reports TOMO Coffee was founded in 2022 and identifies Manila, Philippines as its location. It separately lists Philippine incorporation in 2022. The profile displays an update date of June 27, 2025. Recorded as a reported Philippine founding year, not a confirmed first store opening date. The previously inspected official About page stated Established 2021. This discrepancy remains unresolved."
4,BFC005,TOMORO COFFEE,Partial,SRC0008;SRC0045;SRC0102,<NA>,"User supplied SRC0008 browser text names TOMORO Coffee Philippines Inc. as application provider. Store operator and franchise contracting entity remain unresolved. Terms describe pickup, delivery, and promotional benefits. Features have not been tested. | SRC0102: June 15, 2024 reporting states that TOMORO opened its first wave of Philippine stores in April 2024, followed by its Manila flagship opening on June 13, 2024. Philippine start year is 2024. Flagship opening is not treated as the first Philippine operating date. Historical expansion targets and franchise offers are not accepted as current terms."
11,BFC012,Farron Café,Partial,SRC0055;SRC0108,<NA>,"SRC0055: PFA listing on B Hive Connect names Farron Cafe Ventures Inc. Precise operating role, listing effective date, and current offer unresolved. | PH history review: The Philippine Star reports the first Farron Cafe food cart opened beside UE Manila in February 2010. This records operating history rather than incorporation or franchising."



No dates inferred. No files changed.


In [8]:
# 1. Create separate working copies.

clean_datasets = {}

for dataset_name, df in datasets.items():
    clean_datasets[dataset_name] = df.copy()


# 2. Standardise outer whitespace and empty strings.
# Internal wording and evidence qualifications are preserved.

for dataset_name, df in clean_datasets.items():
    for column in df.columns:
        df[column] = df[column].str.strip().replace("", pd.NA)


# 3. Convert validated numeric fields.
# Stop rather than silently replace a nonnumeric value.

for dataset_name, fields in numeric_fields.items():
    df = clean_datasets[dataset_name]

    for field in fields:
        numbers = pd.to_numeric(df[field], errors="coerce")

        invalid_values = df[field].notna() & numbers.isna()

        if invalid_values.any():
            raise ValueError(
                f"Unresolved numeric values in {dataset_name}: {field}"
            )

        if numbers.isin([float("inf"), -float("inf")]).any():
            raise ValueError(
                f"Nonfinite values in {dataset_name}: {field}"
            )

        if field in whole_number_fields:
            if numbers.dropna().mod(1).ne(0).any():
                raise ValueError(
                    f"Noninteger values in {dataset_name}: {field}"
                )

            df[field] = numbers.astype("Int64")
        else:
            df[field] = numbers.astype("Float64")


# 4. Convert complete calendar dates.
# Missing dates remain missing. Relative labels in notes are untouched.

for dataset_name, fields in date_fields.items():
    df = clean_datasets[dataset_name]

    for field in fields:
        correct_format = df[field].str.fullmatch(
            r"\d{4}-\d{2}-\d{2}"
        ).fillna(False)

        dates = pd.to_datetime(
            df[field].where(correct_format),
            format="%Y-%m-%d",
            errors="coerce",
        )

        invalid_dates = df[field].notna() & dates.isna()

        if invalid_dates.any():
            raise ValueError(
                f"Unresolved date values in {dataset_name}: {field}"
            )

        df[field] = dates


# 5. Keep unresolved record qualifications in a separate review table.

cleaning_review = pd.DataFrame([
    {
        "dataset": "source_register",
        "record_id": "SRC0035",
        "issue": "Menu request redirected to rewards",
        "treatment": (
            "Retain audit record. Not accepted as menu or pricing evidence. "
            "Missing collection metadata remains unresolved."
        ),
    },
    {
        "dataset": "competitor_master",
        "record_id": "BFC004",
        "issue": "Conflicting founding statements and missing verification date",
        "treatment": (
            "Retain reported 2022 year and conflicting 2021 statement. "
            "Keep verification date missing. First store date unconfirmed."
        ),
    },
    {
        "dataset": "competitor_master",
        "record_id": "BFC005",
        "issue": "Missing verification date",
        "treatment": (
            "Retain partial verification and existing source qualifications. "
            "Do not infer a verification date from source collection dates."
        ),
    },
    {
        "dataset": "competitor_master",
        "record_id": "BFC012",
        "issue": "Missing verification date",
        "treatment": (
            "Retain partial verification and existing source qualifications. "
            "Do not infer a verification date from source collection dates."
        ),
    },
])


# 6. Confirm record counts and converted types.

conversion_rows = []

for dataset_name, df in clean_datasets.items():
    if len(df) != len(datasets[dataset_name]):
        raise ValueError(f"Record count changed: {dataset_name}")

    for field in numeric_fields.get(dataset_name, []):
        conversion_rows.append({
            "dataset": dataset_name,
            "field": field,
            "data_type": str(df[field].dtype),
        })

    for field in date_fields.get(dataset_name, []):
        conversion_rows.append({
            "dataset": dataset_name,
            "field": field,
            "data_type": str(df[field].dtype),
        })

print("Converted field types:")
display(pd.DataFrame(conversion_rows))

print("\nUnresolved record qualifications:")
display(cleaning_review)

print("Working datasets prepared:", len(clean_datasets))
print("All record counts preserved.")
print("Raw datasets remain unchanged.")
print("No files saved yet.")

Converted field types:


,dataset,field,data_type
0,competitor_master,global_founding_year,Int64
1,competitor_master,ph_start_year,Int64
2,competitor_master,last_verified_date,datetime64[us]
3,branch_directory,status_date,datetime64[s]
4,branch_directory,observed_date,datetime64[us]
5,menu_pricing,volume_ml,Float64
6,menu_pricing,regular_price_php,Float64
7,menu_pricing,promo_price_php,Float64
8,menu_pricing,displayed_price_php,Float64
9,menu_pricing,observed_date,datetime64[us]



Unresolved record qualifications:


,dataset,record_id,issue,treatment
0,source_register,SRC0035,Menu request redirected to rewards,Retain audit record. Not accepted as menu or p...
1,competitor_master,BFC004,Conflicting founding statements and missing ve...,Retain reported 2022 year and conflicting 2021...
2,competitor_master,BFC005,Missing verification date,Retain partial verification and existing sourc...
3,competitor_master,BFC012,Missing verification date,Retain partial verification and existing sourc...


Working datasets prepared: 12
All record counts preserved.
Raw datasets remain unchanged.
No files saved yet.


In [9]:
# 1. Inspect labels used in important categorical fields.

category_fields = {
    "competitor_master": [
        "inclusion_status",
        "verification_status",
    ],
    "branch_directory": [
        "operating_status",
        "verification_status",
        "store_format",
    ],
    "menu_pricing": [
        "channel",
        "comparison_group",
        "temperature",
        "size_label",
        "price_basis",
        "availability_status",
    ],
    "customer_reviews": [
        "platform",
        "review_scope",
        "text_basis",
        "company_selected",
    ],
    "store_assessments": [
        "observation_method",
    ],
    "ordering_channels": [
        "checkout_verified",
    ],
}

category_tables = []

for dataset_name, fields in category_fields.items():
    df = clean_datasets[dataset_name]

    for field in fields:
        counts = df[field].value_counts(dropna=False).reset_index()
        counts.columns = ["recorded_label", "number_of_records"]
        counts.insert(0, "field", field)
        counts.insert(0, "dataset", dataset_name)
        category_tables.append(counts)

category_summary = pd.concat(category_tables, ignore_index=True)

print("Recorded category labels:")
with pd.option_context("display.max_rows", None):
    display(category_summary)


# 2. Flag menu records with the same recorded comparison details.
# Exclude source_id so separate captures can be inspected together.

menu = clean_datasets["menu_pricing"]

menu_keys = [
    "brand_id",
    "product_name",
    "temperature",
    "size_label",
    "channel",
    "location_scope",
    "observed_date",
    "price_basis",
]

repeated_menu_mask = menu.duplicated(
    subset=menu_keys,
    keep=False,
)

menu_recheck_candidates = menu.loc[
    repeated_menu_mask,
    menu_keys + [
        "observation_id",
        "source_id",
        "displayed_price_php",
        "promo_price_php",
        "regular_price_php",
    ],
].sort_values(
    ["brand_id", "product_name", "source_id"]
)

print("\nMenu records requiring repeat observation review:")
display(menu_recheck_candidates)


# 3. Flag repeated review text within the same brand and platform.
# Temporary text normalisation is used only for detection.

reviews = clean_datasets["customer_reviews"].copy()

reviews["text_check"] = (
    reviews["review_text"]
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

repeated_review_mask = (
    reviews["text_check"].notna()
    & reviews.duplicated(
        subset=["brand_id", "platform", "text_check"],
        keep=False,
    )
)

print("\nRepeated review text requiring inspection:")
display(
    reviews.loc[
        repeated_review_mask,
        [
            "review_id",
            "sample_id",
            "brand_id",
            "branch_id",
            "platform",
            "source_id",
            "review_text",
        ],
    ]
)

print("\nMenu records flagged:", int(repeated_menu_mask.sum()))
print("Review records flagged:", int(repeated_review_mask.sum()))
print("Matching missing fields do not establish duplicate identity.")
print("No records removed, labels changed or files saved.")

Recorded category labels:


,dataset,field,recorded_label,number_of_records
0,competitor_master,inclusion_status,Included,19
1,competitor_master,inclusion_status,Candidate,3
2,competitor_master,inclusion_status,Reference,1
3,competitor_master,verification_status,Partial,22
4,competitor_master,verification_status,Pending,1
5,branch_directory,operating_status,Pricing listing only; current operation unconf...,18
6,branch_directory,operating_status,Listed; current operation unconfirmed,6
7,branch_directory,verification_status,Pending,13
8,branch_directory,verification_status,Partial,11
9,branch_directory,store_format,<NA>,24



Menu records requiring repeat observation review:


,brand_id,product_name,temperature,size_label,channel,location_scope,observed_date,price_basis,observation_id,source_id,displayed_price_php,promo_price_php,regular_price_php
77,BFC013,Chatime Milk Tea,Not stated,<NA>,Foodpanda,"WMall, Pasay City",2026-10-23,Product card starting price,MNU0078,SRC0048,104.0,104.0,<NA>
94,BFC013,Chatime Milk Tea,Not stated,<NA>,Foodpanda,"WMall, Pasay City",2026-10-23,Product card starting price,MNU0095,SRC0056,130.0,<NA>,<NA>
78,BFC013,Pearl Milk Tea,Not stated,<NA>,Foodpanda,"WMall, Pasay City",2026-10-23,Product card starting price,MNU0079,SRC0048,112.0,112.0,<NA>
95,BFC013,Pearl Milk Tea,Not stated,<NA>,Foodpanda,"WMall, Pasay City",2026-10-23,Product card starting price,MNU0096,SRC0056,140.0,<NA>,<NA>
33,BFC022,Caffe Latte,Not stated,<NA>,Foodpanda,"One Ecom Center, Pasay",2026-10-23,Product card starting price,MNU0034,SRC0037,190.0,<NA>,<NA>
107,BFC022,Caffe Latte,Not stated,<NA>,Foodpanda,"One Ecom Center, Pasay",2026-10-23,Product card starting price,MNU0108,SRC0119,190.0,<NA>,<NA>



Repeated review text requiring inspection:


,review_id,sample_id,brand_id,branch_id,platform,source_id,review_text



Menu records flagged: 6
Review records flagged: 0
Matching missing fields do not establish duplicate identity.
No records removed, labels changed or files saved.


In [10]:
from datetime import datetime
from zoneinfo import ZoneInfo
import shutil


# 1. Add a standardised size field while preserving recorded labels.

menu = clean_datasets["menu_pricing"]

size_mapping = {
    "16Oz": "16oz",
    "22Oz": "22oz",
    "32Oz": "32oz",
}

menu["size_label_standardised"] = menu["size_label"].replace(
    size_mapping
)

changed_size = (
    menu["size_label"].notna()
    & menu["size_label"].ne(menu["size_label_standardised"])
).fillna(False)

print("Size labels standardised:")
display(
    menu.loc[
        changed_size,
        [
            "observation_id",
            "product_name",
            "size_label",
            "size_label_standardised",
        ],
    ]
)


# 2. Document the repeated observations without deleting them.

repeat_decisions = pd.DataFrame([
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0078",
        "issue": "Chatime Milk Tea promotional version",
        "treatment": (
            "Retain alongside MNU0095. Different retrieved price versions. "
            "Do not interpret as a dated price change or combine "
            "as independent comparable prices."
        ),
    },
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0095",
        "issue": "Chatime Milk Tea recheck version",
        "treatment": (
            "Retain alongside MNU0078. Effective price and "
            "promotion conditions remain unresolved."
        ),
    },
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0079",
        "issue": "Chatime Pearl Milk Tea promotional version",
        "treatment": (
            "Retain alongside MNU0096. Different retrieved price versions. "
            "Do not interpret as a dated price change or combine "
            "as independent comparable prices."
        ),
    },
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0096",
        "issue": "Chatime Pearl Milk Tea recheck version",
        "treatment": (
            "Retain alongside MNU0079. Effective price and "
            "promotion conditions remain unresolved."
        ),
    },
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0034",
        "issue": "Starbucks Latte supported by another capture",
        "treatment": (
            "Retain alongside MNU0108 for traceability. "
            "Do not count both as independent price observations "
            "in a single comparison."
        ),
    },
    {
        "dataset": "menu_pricing",
        "record_id": "MNU0108",
        "issue": "Starbucks Latte repeat capture",
        "treatment": (
            "Retain alongside MNU0034 for traceability. "
            "Same displayed price does not resolve serving size."
        ),
    },
])

missing_review_ids = set(repeat_decisions["record_id"]) - set(
    menu["observation_id"]
)

if missing_review_ids:
    raise ValueError(
        f"Expected observation IDs not found: {missing_review_ids}"
    )

cleaning_review = pd.concat(
    [cleaning_review, repeat_decisions],
    ignore_index=True,
).drop_duplicates(
    subset=["dataset", "record_id", "issue"]
)


# 3. Validate prepared tables before saving.

for dataset_name, df in clean_datasets.items():
    id_column = dataset_ids[dataset_name]

    if len(df) != len(datasets[dataset_name]):
        raise ValueError(f"Record count changed: {dataset_name}")

    if df[id_column].isna().any():
        raise ValueError(f"Missing identifiers: {dataset_name}")

    if df[id_column].duplicated().any():
        raise ValueError(f"Repeated identifiers: {dataset_name}")


# 4. Prepare a field type register for reproducible loading.
# CSV files do not preserve pandas data types.

field_type_rows = []

for dataset_name, df in clean_datasets.items():
    for column in df.columns:
        field_type_rows.append({
            "dataset": dataset_name,
            "field": column,
            "pandas_dtype": str(df[column].dtype),
        })

field_types = pd.DataFrame(field_type_rows)


# 5. Back up existing prepared outputs before writing.

processed_data_path.mkdir(parents=True, exist_ok=True)

report_path = processed_data_path / "quality_reports"
report_path.mkdir(parents=True, exist_ok=True)

save_time = datetime.now(ZoneInfo("Asia/Manila"))
save_label = save_time.strftime("%Y%m%d_%H%M%S_%f")

backup_path = (
    references_path / "backups" / f"prepared_data_{save_label}"
)

output_paths = [
    processed_data_path / f"{name}.csv"
    for name in clean_datasets
]

output_paths += [
    report_path / "cleaning_review.csv",
    report_path / "field_types.csv",
]

for file_path in output_paths:
    if file_path.is_file():
        relative_path = file_path.relative_to(processed_data_path)
        backup_file = backup_path / relative_path

        backup_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, backup_file)


# 6. Save prepared copies and review documentation.

saved_rows = []

for dataset_name, df in clean_datasets.items():
    output_path = processed_data_path / f"{dataset_name}.csv"

    df.to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d",
    )

    saved_rows.append({
        "dataset": dataset_name,
        "records_saved": len(df),
        "columns_saved": len(df.columns),
    })

cleaning_review.to_csv(
    report_path / "cleaning_review.csv",
    index=False,
)

field_types.to_csv(
    report_path / "field_types.csv",
    index=False,
)

print("\nPrepared datasets saved:")
display(pd.DataFrame(saved_rows))

print("Size labels standardised:", int(changed_size.sum()))
print("Documented review items:", len(cleaning_review))
print("Prepared data:", processed_data_path)
print("Quality reports:", report_path)
print("Raw files were not changed.")
print("No records removed or missing evidence filled.")

Size labels standardised:


,observation_id,product_name,size_label,size_label_standardised
91,MNU0092,Mango Shake 16Oz,16Oz,16oz
92,MNU0093,Mango Shake 22Oz,22Oz,22oz
93,MNU0094,Mango Shake 32Oz,32Oz,32oz



Prepared datasets saved:


,dataset,records_saved,columns_saved
0,competitor_master,23,19
1,branch_directory,24,15
2,menu_pricing,246,22
3,promotions,3,21
4,loyalty_programs,4,16
5,store_assessments,3,21
6,customer_reviews,36,16
7,ordering_channels,6,20
8,franchise_offers,15,27
9,commercial_developments,7,15


Size labels standardised: 3
Documented review items: 10
Prepared data: /Users/jannoelvero/Documents/bfc_compcheck_2026/data/processed
Quality reports: /Users/jannoelvero/Documents/bfc_compcheck_2026/data/processed/quality_reports
Raw files were not changed.
No records removed or missing evidence filled.


In [11]:
# Document supplementary qualifications without deleting records.

from datetime import datetime
from zoneinfo import ZoneInfo
import shutil

# 1. Add a standardised size field while preserving recorded labels.

menu = clean_datasets["menu_pricing"]

parent_listing = menu.loc[
    menu["source_id"].eq("SRC0126")
    & menu["product_name"].eq("Americano"),
    "observation_id",
]

if len(parent_listing) != 1:
    raise ValueError("Expected one SRC0126 Americano listing.")

supplementary_review = pd.DataFrame(
    [
        {
            "dataset": "menu_pricing",
            "record_id": parent_listing.iloc[0],
            "issue": "General listing overlaps with detailed serving options",
            "treatment": (
                "Retain for traceability. Use SRC0127 for explicit iced "
                "Americano variants. Do not count this general listing "
                "as an additional independent variant price."
            ),
        },
        {
            "dataset": "menu_pricing",
            "record_id": "MNU0150",
            "issue": "Latte name but espresso unspecified in description",
            "treatment": (
                "Preserve the published name and description. "
                "Composition remains unresolved in the intermediate "
                "PICKUP profile. Do not infer espresso from the hot variant."
            ),
        },
        {
            "dataset": "source_register",
            "record_id": "SRC0126",
            "issue": "Head Office context and unresolved app messaging",
            "treatment": (
                "Do not assign prices to other branches. Preserve the "
                "app launching soon message alongside existing app "
                "evidence. Download controls were not successfully "
                "activated in the separate browser inspection."
            ),
        },
        {
            "dataset": "source_register",
            "record_id": "SRC0136",
            "issue": "Retrospective extraction from a preserved baseline capture",
            "treatment": (
                "Retain October 23 as the capture and observation date. "
                "Document October 25 as the extraction review date. "
                "Assign food evidence to V Mapa, not Malingap."
            ),
        },
    ]
)

# Confirm that every reviewed record exists.

for _, review in supplementary_review.iterrows():
    dataset_name = review["dataset"]
    id_column = dataset_ids[dataset_name]

    if review["record_id"] not in clean_datasets[dataset_name][id_column].values:
        raise ValueError(
            f"Reviewed record not found: {review['record_id']}"
        )

cleaning_review = pd.concat(
    [cleaning_review, supplementary_review],
    ignore_index=True,
).drop_duplicates(
    subset=["dataset", "record_id", "issue"]
).reset_index(drop=True)

with pd.option_context("display.max_colwidth", None):
    display(supplementary_review)

print("Supplementary review decisions:", len(supplementary_review))
print("Total documented review items:", len(cleaning_review))
print("No records removed or raw values changed.")
print("No files saved by this cell.")

,dataset,record_id,issue,treatment
0,menu_pricing,MNU0207,General listing overlaps with detailed serving options,Retain for traceability. Use SRC0127 for explicit iced Americano variants. Do not count this general listing as an additional independent variant price.
1,menu_pricing,MNU0150,Latte name but espresso unspecified in description,Preserve the published name and description. Composition remains unresolved in the intermediate PICKUP profile. Do not infer espresso from the hot variant.
2,source_register,SRC0126,Head Office context and unresolved app messaging,Do not assign prices to other branches. Preserve the app launching soon message alongside existing app evidence. Download controls were not successfully activated in the separate browser inspection.
3,source_register,SRC0136,Retrospective extraction from a preserved baseline capture,"Retain October 23 as the capture and observation date. Document October 25 as the extraction review date. Assign food evidence to V Mapa, not Malingap."


Supplementary review decisions: 4
Total documented review items: 14
No records removed or raw values changed.
No files saved by this cell.


In [12]:
# Load supplementary catalogs separately from price observations.

catalog_files = {
    "pickup_product_profile": "pickup_product_profile.csv",
    "zus_product_catalog": "zus_product_catalog.csv",
    "zus_food_catalog": "zus_food_catalog.csv",
    "bigbrew_product_catalog": "bigbrew_product_catalog.csv",
}

prepared_catalogs = {}
catalog_check_rows = []

sources = clean_datasets["source_register"]
brands = clean_datasets["competitor_master"]

source_lookup = sources.set_index("source_id")
valid_brand_ids = brands["brand_id"]

for catalog_name, filename in catalog_files.items():
    file_path = project_root / "data" / "interim" / filename
    df = pd.read_csv(file_path, dtype="string")

    # Standardize outer whitespace without altering descriptions.
    for column in df.columns:
        df[column] = df[column].str.strip().replace("", pd.NA)

    required_columns = ["brand_id", "product_name", "source_id"]

    if not set(required_columns).issubset(df.columns):
        raise ValueError(f"Required fields missing: {catalog_name}")

    if df[required_columns].isna().any().any():
        raise ValueError(f"Missing required values: {catalog_name}")

    if df.duplicated(["brand_id", "product_name", "source_id"]).any():
        raise ValueError(f"Repeated product source records: {catalog_name}")

    if not df["brand_id"].isin(valid_brand_ids).all():
        raise ValueError(f"Unknown brand references: {catalog_name}")

    if not df["source_id"].isin(sources["source_id"]).all():
        raise ValueError(
            f"Unknown sources in {catalog_name}. "
            "Reload Notebook 03 after the source register update."
        )

    source_brands = df["source_id"].map(source_lookup["brand_id"])

    if (
        source_brands.isna().any()
        or df["brand_id"].ne(source_brands).any()
    ):
        raise ValueError(f"Source brand agreement unresolved: {catalog_name}")

    # Check the catalog's own evidence file when present.
    # Otherwise use the file linked through its registered source.
    evidence_files = df["source_id"].map(source_lookup["evidence_file"])

    if "evidence_file" in df.columns:
        evidence_files = df["evidence_file"].fillna(evidence_files)

    for evidence_file in evidence_files.drop_duplicates():
        if pd.isna(evidence_file):
            raise ValueError(f"Missing evidence reference: {catalog_name}")

        if not (project_root / evidence_file).is_file():
            raise FileNotFoundError(evidence_file)

    # Validate the separate description sources where populated.
    if "description_source_id" in df.columns:
        description_ids = df["description_source_id"].dropna()

        if not description_ids.isin(sources["source_id"]).all():
            raise ValueError(f"Unknown description sources: {catalog_name}")

        description_brands = df["description_source_id"].map(
            source_lookup["brand_id"]
        )

        populated = df["description_source_id"].notna()

        if (
            description_brands.loc[populated].isna().any()
            or df.loc[populated, "brand_id"].ne(
                description_brands.loc[populated]
            ).any()
        ):
            raise ValueError(
                f"Description source brand mismatch: {catalog_name}"
            )

        for source_id in description_ids.drop_duplicates():
            evidence_file = source_lookup.loc[source_id, "evidence_file"]

            if pd.isna(evidence_file):
                raise ValueError(f"Missing description evidence: {source_id}")

            if not (project_root / evidence_file).is_file():
                raise FileNotFoundError(evidence_file)

    if "observed_date" in df.columns:
        df["observed_date"] = pd.to_datetime(
            df["observed_date"],
            format="%Y-%m-%d",
            errors="raise",
        )

    if "displayed_price_php" in df.columns:
        df["displayed_price_php"] = pd.to_numeric(
            df["displayed_price_php"],
            errors="raise",
        ).astype("Float64")

    prepared_catalogs[catalog_name] = df

    catalog_check_rows.append(
        {
            "catalog": catalog_name,
            "records": len(df),
            "primary_sources": df["source_id"].nunique(),
            "reference_checks": "Passed",
            "evidence_files": "Found",
        }
    )

catalog_validation = pd.DataFrame(catalog_check_rows)

display(catalog_validation)

print("Catalogs prepared:", len(prepared_catalogs))
print("Catalog rows:", sum(len(df) for df in prepared_catalogs.values()))
print("Catalogs remain separate from menu price observations.")
print("Draft classifications and evidence qualifications were retained.")


print("No files saved.")

,catalog,records,primary_sources,reference_checks,evidence_files
0,pickup_product_profile,74,1,Passed,Found
1,zus_product_catalog,36,3,Passed,Found
2,zus_food_catalog,19,4,Passed,Found
3,bigbrew_product_catalog,12,1,Passed,Found


Catalogs prepared: 4
Catalog rows: 141
Catalogs remain separate from menu price observations.
Draft classifications and evidence qualifications were retained.
No files saved.


In [13]:
# Compare the saved source register with Notebook 03's working copy.

saved_sources = pd.read_csv(
    references_path / "source_register.csv",
    dtype="string",
)

zus_catalog = pd.read_csv(
    project_root / "data" / "interim" / "zus_product_catalog.csv",
    dtype="string",
)

catalog_source_ids = zus_catalog["source_id"].drop_duplicates()

source_version_check = pd.DataFrame(
    {
        "source_id": catalog_source_ids,
        "in_saved_register": catalog_source_ids.isin(
            saved_sources["source_id"]
        ),
        "in_working_register": catalog_source_ids.isin(
            clean_datasets["source_register"]["source_id"]
        ),
    }
)

display(source_version_check)

print("Saved source records:", len(saved_sources))
print(
    "Working source records:",
    len(clean_datasets["source_register"]),
)
print("Register path:", references_path / "source_register.csv")
print("No files changed.")

,source_id,in_saved_register,in_working_register
0,SRC0137,True,True
12,SRC0138,True,True
24,SRC0139,True,True


Saved source records: 140
Working source records: 140
Register path: /Users/jannoelvero/Documents/bfc_compcheck_2026/references/source_register.csv
No files changed.


In [14]:
from datetime import datetime
from zoneinfo import ZoneInfo
import shutil

import pandas as pd
from IPython.display import display

# Confirm that the prepared source register is current.

prepared_source_path = (
    project_root / "data" / "processed" / "source_register.csv"
)

prepared_sources = pd.read_csv(
    prepared_source_path,
    dtype="string",
)

working_source_ids = set(
    clean_datasets["source_register"]["source_id"]
)

if set(prepared_sources["source_id"]) != working_source_ids:
    raise ValueError(
        "Export the current prepared datasets before saving catalogs."
    )

# Build a manifest and field type register.

manifest_rows = []
field_type_rows = []

for catalog_name, df in prepared_catalogs.items():
    manifest_rows.append(
        {
            "catalog": catalog_name,
            "filename": f"{catalog_name}.csv",
            "records": len(df),
            "columns": len(df.columns),
            "primary_sources": df["source_id"].nunique(),
        }
    )

    for column in df.columns:
        field_type_rows.append(
            {
                "catalog": catalog_name,
                "field": column,
                "pandas_dtype": str(df[column].dtype),
            }
        )

catalog_manifest = pd.DataFrame(manifest_rows)
catalog_field_types = pd.DataFrame(field_type_rows)

catalogs_path = project_root / "data" / "processed" / "catalogs"
catalogs_path.mkdir(parents=True, exist_ok=True)

outputs = {}

for catalog_name, df in prepared_catalogs.items():
    outputs[f"{catalog_name}.csv"] = df

outputs["catalog_manifest.csv"] = catalog_manifest
outputs["catalog_field_types.csv"] = catalog_field_types

# Preserve existing catalog exports before replacement.

existing_files = [
    catalogs_path / filename
    for filename in outputs
    if (catalogs_path / filename).is_file()
]

if existing_files:
    backup_label = datetime.now(
        ZoneInfo("Asia/Manila")
    ).strftime("%Y%m%d_%H%M%S_%f")

    backup_path = catalogs_path / "backups" / backup_label
    backup_path.mkdir(parents=True, exist_ok=True)

    for file_path in existing_files:
        shutil.copy2(file_path, backup_path / file_path.name)

for filename, table in outputs.items():
    table.to_csv(
        catalogs_path / filename,
        index=False,
        date_format="%Y-%m-%d",
    )

display(catalog_manifest)

print("Catalogs saved:", len(prepared_catalogs))
print("Catalog rows saved:", catalog_manifest["records"].sum())
print("Saved folder:", catalogs_path)
print("Field types and manifest saved.")
print("Draft classifications and qualifications retained.")
print("Raw and intermediate files were not changed.")

,catalog,filename,records,columns,primary_sources
0,pickup_product_profile,pickup_product_profile.csv,74,13,1
1,zus_product_catalog,zus_product_catalog.csv,36,15,3
2,zus_food_catalog,zus_food_catalog.csv,19,9,4
3,bigbrew_product_catalog,bigbrew_product_catalog.csv,12,9,1


Catalogs saved: 4
Catalog rows saved: 141
Saved folder: /Users/jannoelvero/Documents/bfc_compcheck_2026/data/processed/catalogs
Field types and manifest saved.
Draft classifications and qualifications retained.
Raw and intermediate files were not changed.


In [15]:
preparation_summary = pd.DataFrame({
    "measure": [
        "Prepared source records",
        "Prepared menu observations",
        "Prepared catalog rows",
        "Saved cleaning review items"
    ],
    "records": [
        len(clean_datasets["source_register"]),
        len(clean_datasets["menu_pricing"]),
        catalog_manifest["records"].sum(),
        len(pd.read_csv(
            report_path / "cleaning_review.csv",
            dtype="string"
        ))
    ]
})

display(preparation_summary)

,measure,records
0,Prepared source records,140
1,Prepared menu observations,246
2,Prepared catalog rows,141
3,Saved cleaning review items,10


In [16]:
saved_cleaning_review = pd.read_csv(
    report_path / "cleaning_review.csv",
    dtype="string"
)

supplementary_review_check = supplementary_review[
    ["dataset", "record_id", "issue"]
].merge(
    saved_cleaning_review[["dataset", "record_id", "issue"]],
    on=["dataset", "record_id", "issue"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

display(supplementary_review_check)

,dataset,record_id,issue,_merge
0,menu_pricing,MNU0207,General listing overlaps with detailed serving...,left_only
1,menu_pricing,MNU0150,Latte name but espresso unspecified in descrip...,left_only
2,source_register,SRC0126,Head Office context and unresolved app messaging,left_only
3,source_register,SRC0136,Retrospective extraction from a preserved base...,left_only


In [17]:
from datetime import datetime
from zoneinfo import ZoneInfo

review_file = report_path / "cleaning_review.csv"

backup_label = datetime.now(
    ZoneInfo("Asia/Manila")
).strftime("%Y%m%d_%H%M%S_%f")

saved_cleaning_review.to_csv(
    review_file.with_name(f"cleaning_review_backup_{backup_label}.csv"),
    index=False
)

cleaning_review = pd.concat(
    [saved_cleaning_review, supplementary_review],
    ignore_index=True
).drop_duplicates(
    subset=["dataset", "record_id", "issue"]
).reset_index(drop=True)

cleaning_review.to_csv(review_file, index=False)

saved_cleaning_review = pd.read_csv(
    review_file,
    dtype="string"
)

print("Saved cleaning review items:", len(saved_cleaning_review))

Saved cleaning review items: 14


In [18]:
print("Loaded sources:", len(datasets["source_register"]))
print("Loaded menu observations:", len(datasets["menu_pricing"]))
print("Loaded cleaning decisions:", len(cleaning_review))

Loaded sources: 140
Loaded menu observations: 246
Loaded cleaning decisions: 14


In [19]:
# Preserve the complete prepared data for supplementary analysis.
all_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

baseline_date = pd.Timestamp("2026-10-23")
period_rows = []

for name, df in all_datasets.items():
    if "observed_date" in df.columns:
        observed_dates = pd.to_datetime(
            df["observed_date"],
            errors="raise"
        )

        baseline_mask = observed_dates.le(baseline_date)
        supplement_mask = observed_dates.gt(baseline_date)

        datasets[name] = df.loc[baseline_mask].copy()

        period_rows.append({
            "dataset": name,
            "baseline_records": int(baseline_mask.sum()),
            "supplementary_records": int(supplement_mask.sum()),
            "undated_records": int(observed_dates.isna().sum())
        })

evidence_period_inventory = pd.DataFrame(period_rows)

display(evidence_period_inventory)

,dataset,baseline_records,supplementary_records,undated_records
0,branch_directory,24,0,0
1,menu_pricing,131,115,0
2,promotions,1,2,0
3,loyalty_programs,3,1,0
4,store_assessments,3,0,0
5,ordering_channels,5,1,0
6,franchise_offers,15,0,0
7,commercial_developments,7,0,0
8,event_packages,12,0,0
